# Customer Churn Prediction — Analysis Notebook
End-to-end exploratory analysis, model evaluation, and churn-driver review.


In [ ]:
from pathlib import Path
import json, joblib
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
df = pd.read_csv(ROOT / 'data' / 'customer_churn.csv')
df.head()


In [ ]:
print('Shape:', df.shape)
print('Churn rate:', f"{df['churn'].eq('Yes').mean():.1%}")
df.describe(include="all").T


## Exploratory analysis


In [ ]:
churn_by_contract = df.assign(churn_flag=df['churn'].eq('Yes').astype(int)).groupby('contract_type')['churn_flag'].mean().sort_values(ascending=False)
churn_by_contract.plot(kind="bar", title="Churn rate by contract type", ylabel="Churn rate")
plt.show()


In [ ]:
churn_by_sat = df.assign(churn_flag=df['churn'].eq('Yes').astype(int)).groupby('satisfaction_score')['churn_flag'].mean()
churn_by_sat.plot(marker="o", title="Churn rate by satisfaction score", ylabel="Churn rate")
plt.show()


## Model comparison


In [ ]:
comparison = pd.read_csv(ROOT / 'reports' / 'model_comparison.csv')
comparison.sort_values('roc_auc', ascending=False)


## Feature importance


In [ ]:
importance = pd.read_csv(ROOT / 'reports' / 'feature_importance.csv').head(15).sort_values('importance')
importance.plot(kind='barh', x='feature', y='importance', legend=False, title='Top churn drivers', figsize=(9,6))
plt.show()


## Score customers


In [ ]:
model = joblib.load(ROOT / 'models' / 'churn_model.joblib')
sample = pd.read_csv(ROOT / 'data' / 'sample_customers.csv')
sample['churn_probability'] = model.predict_proba(sample.drop(columns=['customer_id']))[:, 1]
sample[['customer_id','churn_probability']].sort_values('churn_probability', ascending=False).head(10)
